In [299]:
# libraries
import pandas as pd

In [300]:
# Zones
bex_zones = pd.read_csv("../data/raw/BORDER_CarrierZoneLocations_20260424T050203.csv")
tge_zones = pd.read_csv("../data/raw/TEXP_CarrierZoneLocations_20260423T014611.csv")

# Rates
bex_buy = pd.read_csv("../data/raw/Rohlig_Border - Rates Effective 23 Apr 2026 (2).csv")
tge_buy = pd.read_csv("../data/raw/TGE Intermodal - Perth - Rates Effective 13 Apr 2026 (4).csv")

# Subset 'From Zones' to relevant states
bex_buy_clean = bex_buy[bex_buy['From Zone'].isin(["Sydney", "Melbourne", "Perth", "Adelaide", "Brisbane"])]
tge_buy_clean = tge_buy[tge_buy['From Zone'].isin(["SYDNEY", "MELBOURNE", "ADELAIDE", "PERTH", "BRISBANE"])]

# Manual Preprocessing

In [301]:
print("BEX 'From Zones'")
print(bex_buy_clean["From Zone"][:2])
print("\nTGE 'From Zones'")
print(tge_buy_clean["From Zone"][:2])

BEX 'From Zones'
0    Adelaide
1    Adelaide
Name: From Zone, dtype: object

TGE 'From Zones'
0    PERTH
1    PERTH
Name: From Zone, dtype: object


In [302]:
# Fixing case to title
tge_buy_clean["From Zone"] = tge_buy_clean["From Zone"].str.title()
tge_buy_clean["To Zone"] = tge_buy_clean["To Zone"].str.title()
print(tge_buy_clean["From Zone"][:2])
print(tge_buy_clean["To Zone"][100:102])

0    Perth
1    Perth
Name: From Zone, dtype: object
100       Brisbane
101    Broken Hill
Name: To Zone, dtype: object


In [303]:
# Drop Columns 
print("BEX Columns:\n")
print(bex_buy_clean.columns)
print("\nTGE Columns:\n")
print(tge_buy_clean.columns)

BEX Columns:

Index(['Carrier', 'Ratecard', 'Service', 'From Zone', 'To Zone', 'Reciprocal',
       'Cubic Conversion', 'Basic', 'Minimum', 'Break Type', 'Item Type',
       'Break From', 'Break To', 'Price', 'Break Type.1', 'Item Type.1',
       'Break From.1', 'Break To.1', 'Price.1', 'Break Type.2', 'Item Type.2',
       'Break From.2', 'Break To.2', 'Price.2'],
      dtype='object')

TGE Columns:

Index(['Carrier', 'Ratecard', 'Service', 'From Zone', 'To Zone', 'Reciprocal',
       'Cubic Conversion', 'Basic', 'Minimum', 'Break Type', 'Item Type',
       'Break From', 'Break To', 'Price', 'Break Type.1', 'Item Type.1',
       'Break From.1', 'Break To.1', 'Price.1', 'Break Type.2', 'Item Type.2',
       'Break From.2', 'Break To.2', 'Price.2', 'Break Type.3', 'Item Type.3',
       'Break From.3', 'Break To.3', 'Price.3', 'Break Type.4', 'Item Type.4',
       'Break From.4', 'Break To.4', 'Price.4'],
      dtype='object')


In [304]:
# Filter the rate cards to the correct - comparable - service type
bex_buy_clean = bex_buy_clean[bex_buy_clean["Service"].isin(["BUL"])]
tge_buy_clean = tge_buy_clean[tge_buy_clean["Service"].isin(["EXPRESS"])]

# Save constants we will need
bex_cubic_conversion = bex_buy_clean['Cubic Conversion'].iloc[0]
tge_cubic_conversion = tge_buy_clean['Cubic Conversion'].iloc[0]
tge_break_type = tge_buy_clean['Break Type'].iloc[0]
bex_service_bulk = tge_buy_clean['Service'].iloc[0]
tge_service = tge_buy_clean['Service'].iloc[0]

# Drop unnecessary columns
bex_buy_clean.drop(columns=['Carrier', 'Ratecard', 'Reciprocal', 'Item Type', 'Item Type.1', 'Break Type.2', 'Cubic Conversion',
                            'Break Type', 'Break Type.1', 'Item Type.2'], axis=1, inplace=True)
tge_buy_clean.drop(columns=['Carrier', 'Ratecard', 'Reciprocal', 'Break Type', 'Item Type', 'Break Type.1', 'Item Type.1', 'Break Type.2',
                            'Item Type.2', 'Break Type.3', 'Item Type.3', 'Break Type.4', 'Item Type.4','Service',
                            'Cubic Conversion'], axis=1, inplace=True)

# Filter for BUL 
bex_buy_clean = bex_buy_clean[bex_buy_clean['Service'] == 'BUL']


# Fix missing values
bex_buy_clean['Break To.2'] = 99999
tge_buy_clean['Break To.4'] = 99999

# Fix misleading entries
bex_buy_clean['Break From'] = bex_buy_clean['Break From'].replace(0, 1)
tge_buy_clean['Break From'] = tge_buy_clean['Break From'].replace(0, 1)

# Rename columns
bex_buy_clean.rename(columns={"Price":"Price.0", "Break From": "Break From.0", "Break To":"Break To.0"}, inplace=True)
tge_buy_clean.rename(columns={"Price":"Price.0", "Break From": "Break From.0", "Break To":"Break To.0"}, inplace=True)

tge_buy_clean

,From Zone,To Zone,Basic,Minimum,Break From.0,Break To.0,Price.0,Break From.1,Break To.1,Price.1,Break From.2,Break To.2,Price.2,Break From.3,Break To.3,Price.3,Break From.4,Break To.4,Price.4
0,Perth,0847-01,26.98,231.0,1,750.0,0.813,751.0,1500.0,0.770,1501.0,3000.0,0.765,3001.0,5000.0,0.761,5001.0,99999,0.727
1,Perth,0886-01,26.98,234.0,1,750.0,0.825,751.0,1500.0,0.784,1501.0,3000.0,0.778,3001.0,5000.0,0.775,5001.0,99999,0.737
2,Perth,2330-44,26.98,166.0,1,750.0,0.554,751.0,1500.0,0.508,1501.0,3000.0,0.486,3001.0,5000.0,0.474,5001.0,99999,0.469
3,Perth,2343-01,26.98,185.0,1,750.0,0.632,751.0,1500.0,0.597,1501.0,3000.0,0.594,3001.0,5000.0,0.571,5001.0,99999,0.571
4,Perth,2360-01,26.98,185.0,1,750.0,0.632,751.0,1500.0,0.597,1501.0,3000.0,0.594,3001.0,5000.0,0.571,5001.0,99999,0.571
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
248,Sydney,Gold Coast,25.60,80.0,1,750.0,0.216,751.0,1500.0,0.185,1501.0,3000.0,0.170,3001.0,5000.0,0.163,5001.0,99999,0.160
249,Sydney,Newcastle,25.60,65.0,1,750.0,0.154,751.0,1500.0,0.109,1501.0,3000.0,0.088,3001.0,5000.0,0.079,5001.0,99999,0.073
250,Sydney,Nsw Zone 6,25.60,82.0,1,750.0,0.222,751.0,1500.0,0.198,1501.0,3000.0,0.196,3001.0,5000.0,0.189,5001.0,99999,0.189
251,Sydney,Perth,25.60,186.0,1,750.0,0.666,751.0,1500.0,0.638,1501.0,3000.0,0.611,3001.0,5000.0,0.598,5001.0,99999,0.583


In [305]:
# Reshape the data from wide to long
bex_buy_clean = pd.wide_to_long(bex_buy_clean, stubnames=["Break From", "Break To", "Price"], i=["From Zone", "To Zone"], j="Tier", sep = '.')
print(bex_buy_clean)

                            Basic   Minimum Service  Break From  Break To  \
From Zone To Zone  Tier                                                     
Adelaide  Adelaide 0      9.52606  69.84740     BUL         1.0     250.0   
                   1      9.52606  69.84740     BUL       251.0    1500.0   
                   2      9.52606  69.84740     BUL      1501.0   99999.0   
          Albury   0      9.52606  69.84740     BUL         1.0     250.0   
                   1      9.52606  69.84740     BUL       251.0    1500.0   
...                           ...       ...     ...         ...       ...   
Sydney    Yass     1     19.05213  76.19811     BUL       251.0    1500.0   
                   2     19.05213  76.19811     BUL      1501.0   99999.0   
          Young    0     19.05213  76.19811     BUL         1.0     250.0   
                   1     19.05213  76.19811     BUL       251.0    1500.0   
                   2     19.05213  76.19811     BUL      1501.0   99999.0   

In [306]:
# Create flattened dataframe for KNN
bex_buy_model = bex_buy_clean.reset_index()
bex_buy_model.drop(columns=['Tier'], inplace=True)
bex_buy_model.head()

,From Zone,To Zone,Basic,Minimum,Service,Break From,Break To,Price
0,Adelaide,Adelaide,9.52606,69.8474,BUL,1.0,250.0,0.22071
1,Adelaide,Adelaide,9.52606,69.8474,BUL,251.0,1500.0,0.12077
2,Adelaide,Adelaide,9.52606,69.8474,BUL,1501.0,99999.0,0.11140
3,Adelaide,Albury,9.52606,69.8474,BUL,1.0,250.0,0.37480
4,Adelaide,Albury,9.52606,69.8474,BUL,251.0,1500.0,0.26964


In [307]:
# Reshape TGE and display the multiindex dataframe first
tge_buy_clean = pd.wide_to_long(tge_buy_clean, stubnames=['Break From', 'Break To', 'Price'], i=['From Zone', 'To Zone'], j="Tier", sep=".")
print(tge_buy_clean)

                               Basic  Minimum  Break From  Break To  Price
From Zone To Zone        Tier                                             
Perth     0847-01        0     26.98    231.0         1.0     750.0  0.813
                         1     26.98    231.0       751.0    1500.0  0.770
                         2     26.98    231.0      1501.0    3000.0  0.765
                         3     26.98    231.0      3001.0    5000.0  0.761
                         4     26.98    231.0      5001.0   99999.0  0.727
...                              ...      ...         ...       ...    ...
Sydney    Sunshine Coast 0     25.60    116.0         1.0     750.0  0.360
                         1     25.60    116.0       751.0    1500.0  0.321
                         2     25.60    116.0      1501.0    3000.0  0.301
                         3     25.60    116.0      3001.0    5000.0  0.293
                         4     25.60    116.0      5001.0   99999.0  0.288

[1265 rows x 5 columns]


In [308]:
# Create flattened TGE dataframe for KNN
tge_buy_model = tge_buy_clean.reset_index().drop(columns=['Tier'])
tge_buy_model

,From Zone,To Zone,Basic,Minimum,Break From,Break To,Price
0,Perth,0847-01,26.98,231.0,1.0,750.0,0.813
1,Perth,0847-01,26.98,231.0,751.0,1500.0,0.770
2,Perth,0847-01,26.98,231.0,1501.0,3000.0,0.765
3,Perth,0847-01,26.98,231.0,3001.0,5000.0,0.761
4,Perth,0847-01,26.98,231.0,5001.0,99999.0,0.727
...,...,...,...,...,...,...,...
1260,Sydney,Sunshine Coast,25.60,116.0,1.0,750.0,0.360
1261,Sydney,Sunshine Coast,25.60,116.0,751.0,1500.0,0.321
1262,Sydney,Sunshine Coast,25.60,116.0,1501.0,3000.0,0.301
1263,Sydney,Sunshine Coast,25.60,116.0,3001.0,5000.0,0.293


In [309]:
bex_buy_model.to_csv('../data/processed/bex_buy_model.csv', index=False)
tge_buy_model.to_csv('../data/processed/tge_buy_model.csv', index=False)

# Automatic Preprocessing Pipeline

In [310]:
import json
constants = {'bex_cubic_conversion': bex_cubic_conversion, 'tge_cubic_conversion': tge_cubic_conversion}
with open('../data/processed/constants.json', 'w') as f:
    json.dump(constants, f)